# NB08 — Deployment Preparation

> **Pipeline position:** NB04 (YOLO11n) → NB05 (YOLO11s) → NB06 (RT-DETR, deferred) → NB07 (Evaluation) → **NB08 (Deployment)**

## What this notebook does

| Section | Purpose |
|---|---|
| 0. Setup | Load best model weights and paths |
| 1. ONNX Export | Export YOLO11s `best.pt` → ONNX format for CPU inference |
| 2. Verify ONNX | Sanity-check the exported ONNX model |
| 3. Select Example Images | Copy representative test images for demo |
| 4. Generate Deployment Files | Write `app.py`, `requirements.txt`, `README.md` for HF Spaces |
| 5. Save Dataset Config | Copy class names and config for the backend |
| 6. Summary | List all deployment artifacts |

## Deployment Architecture

```
┌─────────────────────┐    API calls     ┌──────────────────────────┐
│  Vercel Frontend    │ ◄──────────────► │  HuggingFace Spaces      │
│  (React / static)   │                  │  (Gradio + ONNX Runtime) │
│                     │                  │  YOLO11s ONNX model      │
└─────────────────────┘                  └──────────────────────────┘
         │                                          │
         └──── GitHub (shared platform) ────────────┘
```

**Model selected: YOLO11s** — 93.2% test mAP50, strong minority class performance (>91% AP50 on vest/no-vest)

In [ ]:
# ============================================================
# 0. SETUP
# ============================================================
import json
import shutil
import warnings
from pathlib import Path

import numpy as np
from PIL import Image
from IPython.display import display, Markdown

warnings.filterwarnings('ignore')

# ---- Paths ----
PROJECT_ROOT = Path('../')
DEPLOY_DIR   = PROJECT_ROOT / 'deployment'
RESULTS_DIR  = PROJECT_ROOT / 'results'
DATA_DIR     = PROJECT_ROOT / 'data'
EXAMPLES_DIR = DEPLOY_DIR / 'examples'

DEPLOY_DIR.mkdir(exist_ok=True)
EXAMPLES_DIR.mkdir(exist_ok=True)

CLASS_NAMES = ['hardhat', 'no-hardhat', 'vest', 'no-vest', 'person']

# ---- Auto-detect best YOLO11s run ----
def find_best_run(prefix):
    """Find the latest run dir matching prefix that has best.pt."""
    candidates = []
    for runs_dir in [Path('runs/detect'), PROJECT_ROOT / 'runs' / 'detect']:
        if not runs_dir.is_dir():
            continue
        for p in runs_dir.iterdir():
            if p.name.startswith(prefix) and p.is_dir():
                candidates.append(p)
    with_best = [p for p in candidates if (p / 'weights' / 'best.pt').exists()]
    pool = with_best if with_best else candidates
    if not pool:
        return None
    return max(pool, key=lambda p: p.stat().st_mtime)

YOLO11S_DIR = find_best_run('ppe_yolo11s_1280')
YOLO11N_DIR = find_best_run('ppe_yolo11n_1280')

assert YOLO11S_DIR is not None, 'YOLO11s run not found'

BEST_PT = YOLO11S_DIR / 'weights' / 'best.pt'
assert BEST_PT.exists(), f'best.pt not found: {BEST_PT}'

print(f'Project root   : {PROJECT_ROOT.resolve()}')
print(f'Deploy dir     : {DEPLOY_DIR.resolve()}')
print(f'YOLO11s run    : {YOLO11S_DIR.resolve()}')
print(f'Best weights   : {BEST_PT} ({BEST_PT.stat().st_size / 1e6:.1f} MB)')
print(f'\nDeployment model: YOLO11s (93.2% test mAP50)')

---
## 1. Export YOLO11s to ONNX

ONNX Runtime provides 1.5–3× faster inference than PyTorch on CPU, which is critical for the free-tier HuggingFace Spaces deployment.

In [ ]:
# ============================================================
# 1. ONNX EXPORT
# ============================================================
from ultralytics import YOLO

model = YOLO(str(BEST_PT))

# Export to ONNX
# half=False because free HF Spaces CPU doesn't support FP16 ONNX
# imgsz=640 for deployment (faster inference on CPU vs 1280 used in training)
onnx_path = model.export(
    format='onnx',
    imgsz=640,
    half=False,
    simplify=True,
    opset=17,
)

onnx_path = Path(onnx_path)
print(f'\nONNX exported to: {onnx_path}')
print(f'ONNX file size : {onnx_path.stat().st_size / 1e6:.1f} MB')

In [ ]:
# ============================================================
# 1b. Copy ONNX model to deployment directory
# ============================================================
deploy_onnx = DEPLOY_DIR / 'best.onnx'
shutil.copy2(onnx_path, deploy_onnx)
print(f'ONNX copied to: {deploy_onnx} ({deploy_onnx.stat().st_size / 1e6:.1f} MB)')

# Also copy the .pt weights as a backup / for local GPU inference
deploy_pt = DEPLOY_DIR / 'best.pt'
shutil.copy2(BEST_PT, deploy_pt)
print(f'PT copied to  : {deploy_pt} ({deploy_pt.stat().st_size / 1e6:.1f} MB)')

---
## 2. Verify ONNX Model

Run a quick sanity check: load the ONNX model, run inference on a test image, and verify the output shape and detections.

In [ ]:
# ============================================================
# 2. VERIFY ONNX MODEL
# ============================================================
import onnx
import onnxruntime as ort

# Validate ONNX model structure
onnx_model = onnx.load(str(deploy_onnx))
onnx.checker.check_model(onnx_model)
print('✅ ONNX model structure is valid')

# Check input/output shapes
for inp in onnx_model.graph.input:
    shape = [d.dim_value for d in inp.type.tensor_type.shape.dim]
    print(f'  Input  : {inp.name} → {shape}')
for out in onnx_model.graph.output:
    shape = [d.dim_value for d in out.type.tensor_type.shape.dim]
    print(f'  Output : {out.name} → {shape}')

del onnx_model  # free memory

In [ ]:
# ============================================================
# 2b. Test inference with ONNX Runtime via Ultralytics
# ============================================================
# Ultralytics can use ONNX models directly
onnx_model_ul = YOLO(str(deploy_onnx))

# Find a test image
test_img_dir = DATA_DIR / 'processed' / 'images' / 'test'
test_images = sorted(test_img_dir.glob('*.jpg'))[:5]

if test_images:
    test_img = test_images[0]
    results = onnx_model_ul(str(test_img), imgsz=640, conf=0.25, verbose=False)
    
    n_detections = len(results[0].boxes)
    print(f'\n✅ ONNX inference successful on: {test_img.name}')
    print(f'   Detections: {n_detections}')
    
    for box in results[0].boxes:
        cls_name = CLASS_NAMES[int(box.cls)]
        conf = float(box.conf)
        print(f'   → {cls_name}: {conf:.2%}')
else:
    print('⚠️ No test images found — skipping inference test')

del onnx_model_ul

---
## 3. Select Example Images for Demo

Copy a handful of representative test images to `deployment/examples/` for the Gradio interface.

In [ ]:
# ============================================================
# 3. SELECT EXAMPLE IMAGES
# ============================================================
import random

random.seed(42)

test_img_dir = DATA_DIR / 'processed' / 'images' / 'test'
all_test_imgs = sorted(test_img_dir.glob('*.jpg'))
print(f'Total test images available: {len(all_test_imgs)}')

# Select 3 diverse example images
# Pick from different parts of the dataset for variety
n_examples = min(3, len(all_test_imgs))
selected = random.sample(all_test_imgs, n_examples)

# Clear old examples
for old in EXAMPLES_DIR.glob('*'):
    if old.name != '.gitkeep':
        old.unlink()

# Copy selected examples
example_paths = []
for i, img_path in enumerate(selected, 1):
    dest = EXAMPLES_DIR / f'example{i}.jpg'
    shutil.copy2(img_path, dest)
    example_paths.append(dest)
    print(f'  Copied: {img_path.name} → {dest.name}')

print(f'\n{len(example_paths)} example images saved to {EXAMPLES_DIR}')

---
## 4. Generate Deployment Files

Write all files needed for the HuggingFace Spaces backend.

In [ ]:
# ============================================================
# 4a. Write app.py — Gradio backend for HF Spaces
# ============================================================
app_code = '''import gradio as gr
from ultralytics import YOLO
from PIL import Image
import numpy as np
import json
import os

# ---- Load model ----
MODEL_PATH = os.path.join(os.path.dirname(__file__), "best.onnx")
model = YOLO(MODEL_PATH)

CLASS_NAMES = ["hardhat", "no-hardhat", "vest", "no-vest", "person"]
VIOLATION_CLASSES = {"no-hardhat", "no-vest"}
COMPLIANT_CLASSES = {"hardhat", "vest"}


def detect_ppe(image, conf_threshold=0.25):
    """Run PPE detection on an uploaded image."""
    if image is None:
        return None, "No image provided."

    # Run inference
    results = model(image, imgsz=640, conf=conf_threshold, verbose=False)
    result = results[0]

    # Draw annotated image
    annotated = result.plot()
    annotated_rgb = Image.fromarray(annotated[..., ::-1])

    # Build compliance summary
    detections = result.boxes
    violations = []
    compliant = []
    persons = 0

    for box in detections:
        cls_name = CLASS_NAMES[int(box.cls)]
        conf = float(box.conf)

        if cls_name in VIOLATION_CLASSES:
            violations.append(f"{cls_name} ({conf:.0%})")
        elif cls_name in COMPLIANT_CLASSES:
            compliant.append(f"{cls_name} ({conf:.0%})")
        elif cls_name == "person":
            persons += 1

    summary = ""
    if violations:
        summary += f"⚠️ VIOLATIONS DETECTED ({len(violations)}):\n"
        for v in violations:
            summary += f"  ❌ {v}\n"
        summary += "\n"
    else:
        summary += "✅ No PPE violations detected.\n\n"

    if compliant:
        summary += f"PPE Compliant Items ({len(compliant)}):\n"
        for c in compliant:
            summary += f"  ✅ {c}\n"
        summary += "\n"

    summary += f"Workers detected: {persons}\n"
    summary += f"Total detections: {len(detections)}"

    # Build JSON result for API consumers (Vercel frontend)
    api_result = {
        "violations": violations,
        "compliant": compliant,
        "persons": persons,
        "total_detections": len(detections),
        "boxes": [],
    }
    for box in detections:
        api_result["boxes"].append({
            "class": CLASS_NAMES[int(box.cls)],
            "confidence": round(float(box.conf), 4),
            "bbox": box.xyxy[0].tolist(),
        })

    return annotated_rgb, summary


# ---- Build Gradio interface ----
example_dir = os.path.join(os.path.dirname(__file__), "examples")
examples = []
if os.path.isdir(example_dir):
    for f in sorted(os.listdir(example_dir)):
        if f.lower().endswith((".jpg", ".jpeg", ".png")):
            examples.append([os.path.join(example_dir, f)])

demo = gr.Interface(
    fn=detect_ppe,
    inputs=[
        gr.Image(type="numpy", label="Upload Construction Site Image"),
        gr.Slider(
            minimum=0.1, maximum=0.9, value=0.25, step=0.05,
            label="Confidence Threshold",
        ),
    ],
    outputs=[
        gr.Image(label="Detection Result"),
        gr.Textbox(label="Compliance Summary", lines=10),
    ],
    title="🦺 PPE Compliance Detector",
    description=(
        "Upload a construction site image to detect hard hats and safety vests. "
        "The model identifies PPE violations (missing hard hat or vest) and "
        "highlights them with bounding boxes.\\n\\n"
        "**Model:** YOLO11s (ONNX) — 93.2% test mAP50 | "
        "**Classes:** hardhat, no-hardhat, vest, no-vest, person"
    ),
    examples=examples if examples else None,
    cache_examples=False,
)

demo.launch()
'''

app_path = DEPLOY_DIR / 'app.py'
app_path.write_text(app_code, encoding='utf-8')
print(f'✅ app.py written to {app_path}')
print(f'   Size: {app_path.stat().st_size:,} bytes')

In [ ]:
# ============================================================
# 4b. Write requirements.txt for HF Spaces
# ============================================================
req_text = '''# PPE Compliance Detector — HuggingFace Spaces
# Runtime: Python 3.11, CPU only
ultralytics>=8.3.50
gradio>=5.12.0
onnxruntime>=1.20.1
opencv-python-headless>=4.10.0
Pillow>=11.1.0
numpy<2.0
'''

req_path = DEPLOY_DIR / 'requirements.txt'
req_path.write_text(req_text, encoding='utf-8')
print(f'✅ requirements.txt written to {req_path}')

In [ ]:
# ============================================================
# 4c. Write HF Spaces README.md (Space metadata)
# ============================================================
readme_text = '''---
title: PPE Compliance Detector
emoji: 🦺
colorFrom: yellow
colorTo: orange
sdk: gradio
sdk_version: 5.12.0
app_file: app.py
pinned: false
license: mit
---

# 🦺 PPE Compliance Detector

Detect whether construction site workers are wearing required Personal Protective Equipment
(hard hats and high-visibility vests) from images.

## Model

- **Architecture:** YOLO11s (Ultralytics)
- **Format:** ONNX (CPU-optimised)
- **Training:** 100 epochs, imgsz=1280, RTX 4060
- **Test mAP50:** 93.2%
- **Minority class AP50:** >91% (vest, no-vest)

## Classes

| Class | Description |
|---|---|
| `hardhat` | Worker wearing a hard hat |
| `no-hardhat` | Worker without a hard hat ⚠️ |
| `vest` | Worker wearing a high-vis vest |
| `no-vest` | Worker without a high-vis vest ⚠️ |
| `person` | Full body of a worker |

## Training Data

Merged from three public datasets (~10K images):
- Construction Site Safety (Roboflow)
- SHWD — Safety Helmet Wearing Dataset (GitHub)
- Pictor-PPE (GitHub)

## Usage

Upload a construction site image and the model will:
1. Detect all workers and PPE items
2. Draw bounding boxes with class labels
3. Generate a compliance summary highlighting violations
'''

readme_path = DEPLOY_DIR / 'README.md'
readme_path.write_text(readme_text, encoding='utf-8')
print(f'✅ README.md written to {readme_path}')

---
## 5. Save Dataset Config & Class Metadata

Save class names and training metadata for the frontend/backend to reference.

In [ ]:
# ============================================================
# 5. SAVE CONFIG FILES
# ============================================================

# 5a. Class names JSON (for frontend)
class_config = {
    'class_names': CLASS_NAMES,
    'num_classes': len(CLASS_NAMES),
    'violation_classes': ['no-hardhat', 'no-vest'],
    'compliant_classes': ['hardhat', 'vest'],
    'model': {
        'architecture': 'YOLO11s',
        'format': 'ONNX',
        'input_size': 640,
        'training_imgsz': 1280,
        'test_map50': 0.932,
        'test_map5095': 0.639,
    },
}

config_path = DEPLOY_DIR / 'model_config.json'
with open(config_path, 'w') as f:
    json.dump(class_config, f, indent=2)
print(f'✅ model_config.json written to {config_path}')

# 5b. Copy training args for reproducibility
args_src = YOLO11S_DIR / 'args.yaml'
if args_src.exists():
    args_dst = DEPLOY_DIR / 'training_args.yaml'
    shutil.copy2(args_src, args_dst)
    print(f'✅ training_args.yaml copied to {args_dst}')

# 5c. Copy hyperparameters
hyp_src = DATA_DIR / 'hyp_ppe.yaml'
if hyp_src.exists():
    hyp_dst = DEPLOY_DIR / 'hyp_ppe.yaml'
    shutil.copy2(hyp_src, hyp_dst)
    print(f'✅ hyp_ppe.yaml copied to {hyp_dst}')

---
## 6. Deployment Artifacts Summary

In [ ]:
# ============================================================
# 6. SUMMARY — List all deployment artifacts
# ============================================================
print('=' * 70)
print('  DEPLOYMENT ARTIFACTS')
print('=' * 70)
print(f'  Directory: {DEPLOY_DIR.resolve()}')
print('-' * 70)

total_size = 0
for f in sorted(DEPLOY_DIR.rglob('*')):
    if f.is_file() and f.name != '.gitkeep':
        size = f.stat().st_size
        total_size += size
        rel = f.relative_to(DEPLOY_DIR)
        if size > 1e6:
            print(f'  {str(rel):<40} {size/1e6:>8.1f} MB')
        elif size > 1024:
            print(f'  {str(rel):<40} {size/1024:>8.1f} KB')
        else:
            print(f'  {str(rel):<40} {size:>8} B')

print('-' * 70)
print(f'  Total: {total_size / 1e6:.1f} MB')
print('=' * 70)

In [ ]:
# ============================================================
# 6b. DEPLOYMENT TREE
# ============================================================
print('\nHuggingFace Spaces structure:')
print('  deployment/')
print('  ├── README.md              # HF Space metadata (sdk: gradio)')
print('  ├── app.py                 # Gradio interface')
print('  ├── requirements.txt       # Python dependencies')
print('  ├── best.onnx              # YOLO11s model (ONNX format)')
print('  ├── best.pt                # YOLO11s model (PyTorch, backup)')
print('  ├── model_config.json      # Class names + model metadata')
print('  ├── training_args.yaml     # Training configuration')
print('  ├── hyp_ppe.yaml           # Augmentation hyperparameters')
print('  └── examples/')
print('      ├── example1.jpg')
print('      ├── example2.jpg')
print('      └── example3.jpg')

In [ ]:
# ============================================================
# 6c. NEXT STEPS
# ============================================================
print('=' * 70)
print('  NEXT STEPS')
print('=' * 70)
print()
print('  1. HuggingFace Spaces (Backend):')
print('     - Create a new HF Space (Gradio SDK)')
print('     - Upload all files from deployment/ directory')
print('     - The ONNX model runs on free CPU tier')
print()
print('  2. Vercel Frontend:')
print('     - Create React/static frontend')
print('     - Point API calls to the HF Spaces Gradio endpoint')
print('     - Use model_config.json for class names/colors')
print()
print('  3. GitHub:')
print('     - Push both repos to GitHub')
print('     - Reference each other in READMEs')
print()
print('  Deployment cost: $0 (same as customer complaint triage)')
print('=' * 70)

---
## Conclusion

All deployment artifacts are now saved and ready:

| Artifact | Purpose |
|---|---|
| `best.onnx` | YOLO11s model in ONNX format for CPU inference on HF Spaces |
| `best.pt` | PyTorch weights backup for local GPU inference |
| `app.py` | Gradio backend with PPE detection + compliance summary |
| `requirements.txt` | Minimal dependencies for HF Spaces |
| `README.md` | HF Space metadata card |
| `model_config.json` | Class names + model metadata for frontend |
| `training_args.yaml` | Full training configuration for reproducibility |
| `hyp_ppe.yaml` | Augmentation hyperparameters |
| `examples/` | 3 representative test images for the demo |

**Architecture:** HuggingFace Spaces (Gradio + ONNX Runtime) ← API → Vercel Frontend, with GitHub as the shared platform. Zero cost deployment.